In [ ]:
from typing import Dict
from collections import defaultdict
from tqdm.notebook import tqdm

import numpy

from scipy.stats import gaussian_kde
from sklearn.metrics import (
    root_mean_squared_error,
    mean_absolute_error,
    r2_score,
)
import torch

from robat.models import RoBaTForNextCyclePrediction
from robat.datasets.ds_preprocess import (
    ds_preprocess_for_eval,
)

from accelerate import Accelerator

from matplotlib import pyplot

In [ ]:
model = RoBaTForNextCyclePrediction.from_pretrained(
    "Junhyeong86/battery-health-prediction-model",
    subfolder="next-cycle-prediction"
)

In [ ]:
ds = ds_preprocess_for_eval(
    ds_name="Inseop97/battery-health-prediction",
    model_purpose="next_cycle_prediction",
    random_time_len=model.config.random_time_len,
    split_cycle_len=256,
)

In [ ]:
accelerator = Accelerator()
model = accelerator.prepare(model)
model.eval()
progress_bar = tqdm(total=len(ds), desc="Evaluating")
overall_label_and_pred = defaultdict(lambda: defaultdict(list))
for cell_id, data in ds.items():
	ref = cell_id.split("/")[0]
	progress_bar.set_postfix({"processing cell id": cell_id})
	inp = {
		"first_discharge_capacity": data["first_discharge_capacity"][:],
		"state_of_health": data["state_of_health"][:],
		"time": data["time"][:],
		"voltage_per_time": data["voltage_per_time"][:],
		"current_per_time": data["current_per_time"][:],
		"mask": data["mask"][:],
		"label_discharge_capacity": data["label_discharge_capacity"][:],
		"label_state_of_health": data["label_state_of_health"][:],
	}
	inp = {k: v.to(accelerator.device) for k, v in inp.items()}
	with torch.no_grad():
		pred = model(**inp)
	mask = data["label_discharge_capacity"][:] == model.config.padding_value
	pred_discharge_capacity = (
		pred.discharge_capacity[:][~mask]
		.detach()
		.cpu()
		.numpy()
	)
	label_discharge_capacity = (
		data["label_discharge_capacity"][:][~mask]
		.detach()
		.cpu()
		.numpy()
	)
	pred_soh = pred_discharge_capacity / data["first_discharge_capacity"][0].item()
	label_soh = label_discharge_capacity / data["first_discharge_capacity"][0].item()

	overall_label_and_pred[ref]["cell_id"].append(cell_id)
	overall_label_and_pred[ref]["pred_discharge_capacity"].append(pred_discharge_capacity)
	overall_label_and_pred[ref]["label_discharge_capacity"].append(label_discharge_capacity)
	overall_label_and_pred[ref]["pred_state_of_health"].append(pred_soh)
	overall_label_and_pred[ref]["label_state_of_health"].append(label_soh)
	progress_bar.update(1)
progress_bar.close()

In [ ]:
ncols=3
nrows = len(overall_label_and_pred) // ncols + (1 if len(overall_label_and_pred) % ncols != 0 else 0)
plot_dict = {
	"discharge_capacity": "Discharge Capacity (Ah)",
}

get_xrange = lambda x: (
    numpy.floor(x.min() * 10) / 10,
    numpy.ceil(x.max() * 10) / 10
)

In [ ]:
for column, column_name in plot_dict.items():
	fig, axes = pyplot.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 4, nrows * 4))
	axes = axes.flatten()
	fig.subplots_adjust(wspace=0.5, hspace=0.5)
	for ax, (ref, data) in zip(axes, overall_label_and_pred.items()):
		ax.set_title(ref)
		ax.set_xlabel("Cycle")
		ax.set_ylabel(column_name)

		ref_iterator = zip(
			data[f"pred_{column}"],
			data[f"label_{column}"],
		)
		for pred, label in ref_iterator:
			cycle = numpy.arange(len(pred))
			# ax.plot(cycle, label, label="ground truth", c="#CC79A7", linestyle="--", zorder=2)
			ax.scatter(cycle, label, label="ground truth", c="#CC79A7", zorder=2, marker="o", s=3)
			ax.scatter(cycle, pred, label="prediction", c="#0072B2", zorder=1, marker="o", s=3)

	handles, labels = ax.get_legend_handles_labels()
	by_label = dict(zip(labels, handles))
	ax.legend(by_label.values(), by_label.keys(), loc="lower left")
	pyplot.show()
	pyplot.close(fig)

In [ ]:
for column, column_name in plot_dict.items():
	fig, axes = pyplot.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 4, nrows * 4))
	axes = axes.flatten()
	fig.subplots_adjust(wspace=0.5, hspace=0.5)
	for ax, (ref, data) in zip(axes, overall_label_and_pred.items()):
		ax.set_title(ref)
		ax.set_xlabel("Cycle")
		ax.set_ylabel(column_name)

		pred = numpy.hstack(data[f"pred_{column}"])
		label = numpy.hstack(data[f"label_{column}"])
		label_and_pred = numpy.vstack((label, pred))
		kde = gaussian_kde(label_and_pred)
		kde_values = kde(label_and_pred)
		kde_values = kde_values / kde_values.max()  # Normalize KDE values for better visualization
		sc = ax.scatter(
			label_and_pred[0],
			label_and_pred[1],
			c=kde_values,
			cmap="viridis",
			s=1,
			zorder=2,
		)
		xrange = get_xrange(label_and_pred[0])
		yrange = get_xrange(label_and_pred[1])
		ax.set_xlim(xrange)
		ax.set_ylim(yrange)
		ax.plot(
			xrange,
			xrange,
			c="red",
			linestyle="--",
			label="y=x",
			zorder=1,
		)
		ax.set_title(f"{ref} label vs prediction")
		ax.set_xlabel(f"Ground Truth {column_name}")
		ax.set_ylabel(f"Predicted {column_name}")
	fig.colorbar(sc, ax=ax, label="KDE Value")
	pyplot.show()
	pyplot.close(fig)

In [ ]:
metrics: Dict[str, Dict[str, float]] = {}
for ref, ref_label_and_pred in overall_label_and_pred.items():
	# Calculate metrics
	dq_rmse = root_mean_squared_error(
		y_true=numpy.hstack(ref_label_and_pred["label_discharge_capacity"]),
		y_pred=numpy.hstack(ref_label_and_pred["pred_discharge_capacity"])
	)
	dq_mae = mean_absolute_error(
		y_true=numpy.hstack(ref_label_and_pred["label_discharge_capacity"]),
		y_pred=numpy.hstack(ref_label_and_pred["pred_discharge_capacity"])
	)
	dq_r2 = r2_score(
		y_true=numpy.hstack(ref_label_and_pred["label_discharge_capacity"]),
		y_pred=numpy.hstack(ref_label_and_pred["pred_discharge_capacity"])
	)
	metrics[ref] ={
		"discharge_capacity RMSE (mAh)": dq_rmse * 1e3,
		"discharge_capacity MAE (mAh)": dq_mae * 1e3,
		"discharge_capacity R2 (%)": dq_r2 * 100,
	}
	soh_rmse = root_mean_squared_error(
		y_true=numpy.hstack(ref_label_and_pred["label_state_of_health"]),
		y_pred=numpy.hstack(ref_label_and_pred["pred_state_of_health"])
	)
	soh_mae = mean_absolute_error(
		y_true=numpy.hstack(ref_label_and_pred["label_state_of_health"]),
		y_pred=numpy.hstack(ref_label_and_pred["pred_state_of_health"])
	)
	soh_r2 = r2_score(
		y_true=numpy.hstack(ref_label_and_pred["label_state_of_health"]),
		y_pred=numpy.hstack(ref_label_and_pred["pred_state_of_health"])
	)
	metrics[ref].update({
		"state_of_health RMSE (%)": soh_rmse * 100,
		"state_of_health MAE (%)": soh_mae * 100,
		"state_of_health R2 (%)": soh_r2 * 100,
	})

In [ ]:
for ref, ref_metrics in metrics.items():
	dq_rmse = ref_metrics["discharge_capacity RMSE (mAh)"]
	dq_mae = ref_metrics["discharge_capacity MAE (mAh)"]
	dq_r2 = ref_metrics["discharge_capacity R2 (%)"]

	print(f"--------- {ref}---------")

	print(
		f"{ref} - Discharge Capacity\n"
		f"- RMSE: {dq_rmse:.4f} mAh\n"
		f"- MAE: {dq_mae:.4f} mAh\n"
		f"- R2: {dq_r2:.4f} %"
	)

	soh_rmse = ref_metrics["state_of_health RMSE (%)"]
	soh_mae = ref_metrics["state_of_health MAE (%)"]
	soh_r2 = ref_metrics["state_of_health R2 (%)"]
	print(
		f"{ref} - State of Health\n"
		f"- RMSE: {soh_rmse:.4f} %\n"
		f"- MAE: {soh_mae:.4f} %\n"
		f"- R2: {soh_r2:.4f} %"
	)

	print("\n\n")